# 01 — Data Exploration
Explore all five datasets used in MindForge.

Datasets:
- `Alpie-core_medical_psychology_dataset.json` — Psychology Q&A with Chain-of-Thought
- `cleanData.csv` — Mental health statement classification
- `Combined Data.csv` — Demographic + health metrics
- `mental_health_dataset.csv` — Therapy Q&A pairs
- `train.csv` — Training split of demographic data

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)

ROOT = Path('..')

## 1. Psychology JSON

In [ ]:
with open(ROOT / 'Alpie-core_medical_psychology_dataset.json', encoding='utf-8') as f:
    psych = json.load(f)

records = psych if isinstance(psych, list) else next((v for v in psych.values() if isinstance(v, list)), [])
print(f'Records: {len(records):,}')
print('Sample keys:', list(records[0].keys()))
print('\nSample prompt:', records[0]['prompt'][:200])

In [ ]:
# Prompt length distribution
prompt_lens = [len(r.get('prompt', '')) for r in records[:5000]]
resp_lens   = [len(r.get('response', '')) for r in records[:5000]]

fig, axes = plt.subplots(1, 2)
axes[0].hist(prompt_lens, bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Prompt length (chars)')
axes[1].hist(resp_lens, bins=40, color='coral', edgecolor='white')
axes[1].set_title('Response length (chars)')
plt.tight_layout()
plt.show()

## 2. Clean Statements (Classification)

In [ ]:
clean_df = pd.read_csv(ROOT / 'cleanData.csv')
print(clean_df.shape)
print(clean_df.dtypes)
clean_df.head()

In [ ]:
# Label distribution
label_col = clean_df.columns[2]  # 'status'
counts = clean_df[label_col].value_counts()
ax = counts.plot(kind='barh', color='steelblue')
ax.set_title('Mental Health Status Distribution (cleanData.csv)')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

## 3. Combined Metrics (Structured Data)

In [ ]:
combined = pd.read_csv(ROOT / 'Combined Data.csv')
print(combined.shape)
combined.head()

In [ ]:
# Risk level distribution
risk_col = 'mental_health_risk'
if risk_col in combined.columns:
    combined[risk_col].value_counts().plot(kind='pie', autopct='%1.1f%%', startangle=90)
    plt.title('Mental Health Risk Distribution')
    plt.ylabel('')
    plt.show()

In [ ]:
# Correlation heatmap
num_cols = combined.select_dtypes(include='number').columns
corr = combined[num_cols].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Therapy Q&A Dataset

In [ ]:
therapy = pd.read_csv(ROOT / 'mental_health_dataset.csv')
print(therapy.shape)
therapy.head(2)

In [ ]:
# Context and response length distributions
therapy['ctx_len'] = therapy.iloc[:, 0].str.len()
therapy['resp_len'] = therapy.iloc[:, 1].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
therapy['ctx_len'].hist(bins=40, ax=axes[0], color='purple', alpha=0.7)
axes[0].set_title('Context length distribution')
therapy['resp_len'].hist(bins=40, ax=axes[1], color='green', alpha=0.7)
axes[1].set_title('Response length distribution')
plt.tight_layout()
plt.show()

## 5. Summary Statistics
Quick overview of all dataset sizes.

In [ ]:
summary = {
    'Psychology JSON': len(records),
    'Clean Statements': len(clean_df),
    'Combined Metrics': len(combined),
    'Therapy Q&A': len(therapy),
}

pd.Series(summary, name='rows').plot(kind='bar', color='indigo', rot=20)
plt.title('Dataset Sizes')
plt.ylabel('Number of rows')
plt.tight_layout()
plt.show()

print('Total training examples (approx):', sum(summary.values()))